## Setup

`.env` must contain the following values:
```
SERVER_IP=your_server_ip
FTP_PORT=21
FTP_USER=your_username
FTP_PASSWORD=your_password
```

Use the same `.venv` you use for dev here with python >= 3.12.10

In [1]:
from ftplib import FTP
from pathlib import Path
from dotenv import load_dotenv
import os

# --- LOAD ENV ---
BASE_DIR = Path().resolve().parents[2] / "Internomat"
env_path = BASE_DIR / ".env"

load_dotenv(env_path)

# --- ENV VARS ---
FTP_HOST = os.getenv("SERVER_IP")
FTP_PORT = int(os.getenv("FTP_PORT", 21))
FTP_USER = os.getenv("FTP_USER")
FTP_PASSWORD = os.getenv("FTP_PASSWORD")

print("\n[ENV DEBUG]")
print("FTP_HOST =", FTP_HOST)
print("FTP_PORT =", FTP_PORT)
print("FTP_USER =", FTP_USER)
print("FTP_PASSWORD =", "SET" if FTP_PASSWORD else "MISSING")

# --- HARD VALIDATION ---
assert FTP_HOST, "FTP_HOST missing (check .env)"
assert FTP_USER, "FTP_USER missing (check .env)"
assert FTP_PASSWORD, "FTP_PASSWORD missing (check .env)"

# --- PATHS ---
REMOTE_DIR = "/cs2/game/csgo/MatchZy"
LOCAL_DIR = BASE_DIR / "test_demos"  

LOCAL_DIR.mkdir(parents=True, exist_ok=True)

print(f"\n[FTP] Connecting to {FTP_HOST}:{FTP_PORT}...")

# --- CONNECT ---
ftp = FTP()

try:
    ftp.connect(FTP_HOST, FTP_PORT, timeout=10)
    ftp.login(FTP_USER, FTP_PASSWORD)
    ftp.set_pasv(True) 

    print("[FTP] Connected")

    # --- NAVIGATE ---
    ftp.cwd(REMOTE_DIR)
    print(f"[FTP] Changed dir → {REMOTE_DIR}")

    files = ftp.nlst()

    downloaded = 0
    skipped = 0

    print(f"[FTP] Found {len(files)} files")

    from tqdm import tqdm

    for file in files:
        if not file.endswith(".dem"):
            continue

        local_file = LOCAL_DIR / file

        if local_file.exists():
            print(f"[SKIP] {file}")
            skipped += 1
            continue

        print(f"[DOWNLOAD] {file}")

        try:
            # --- GET FILE SIZE ---
            filesize = ftp.size(file)

            with open(local_file, "wb") as f, tqdm(
                total=filesize,
                unit="B",
                unit_scale=True,
                desc=file,
                leave=False
            ) as pbar:

                def callback(data):
                    f.write(data)
                    pbar.update(len(data))

                ftp.retrbinary(f"RETR {file}", callback)

            downloaded += 1

        except Exception as e:
            print(f"[ERROR] Failed to download {file}: {e}")

    print("\n[FTP] Done")
    print(f"Downloaded: {downloaded}")
    print(f"Skipped: {skipped}")

except Exception as e:
    print(f"\n FTP ERROR: {e}")

finally:
    try:
        ftp.quit()
    except:
        pass


[ENV DEBUG]
FTP_HOST = 83.141.26.49
FTP_PORT = 21
FTP_USER = server18681
FTP_PASSWORD = SET

[FTP] Connecting to 83.141.26.49:21...
[FTP] Connected
[FTP] Changed dir → /cs2/game/csgo/MatchZy
[FTP] Found 12 files
[SKIP] 2026-03-14_21-36-41_19_cs_alpine_team_prinoX_vs_team_Amü.dem
[SKIP] 2026-03-14_22-28-27_20_de_cache_team_KeTaNoS_vs_team_Tundra_ツ.dem
[DOWNLOAD] 2026-03-14_22-37-47_21_de_cache_team_Amü_vs_team_Tundra_ツ.dem


[DOWNLOAD] 2026-03-14_23-34-54_22_de_cbble_d_team_Dude_vs_team_Sindilia.dem


[DOWNLOAD] 2026-03-15_00-20-53_23_de_dust2_team_Luffy-Eyss_vs_team_Amü.dem


[DOWNLOAD] 2026-03-18_20-41-07_24_de_cbble_d_team_kfinimpar_vs_team_Tundra_ツ.dem


[SKIP] 2026-03-18_21-50-33_25_de_overpass_team_Captain_Blake_vs_team_Dottersack.dem
[SKIP] 2026-03-19_21-10-26_-1_de_thera_team_Captain_Blake_vs_team_Tundra_ツ.dem
[SKIP] 2026-03-19_21-55-45_1_de_warden_team_Dude_vs_team_Amü.dem
[SKIP] 2026-03-20_20-40-16_2_de_thera_team_Soulcrusher_vs_team_Tundra_ツ.dem
[SKIP] 2026-03-20_21-57-48_3_de_mills_team_Sindilia_vs_team_Dottersack.dem
[SKIP] 2026-03-20_22-48-10_4_cs_office_team_TEAMsl1χχ__єχχтяємє_vs_team_Tundra_ツ.dem

[FTP] Done
Downloaded: 4
Skipped: 8


In [ ]:
from pathlib import Path
from awpy import Demo

# from LOCAL_DIR select the first .dem file
DEMO_PATH = next(LOCAL_DIR.glob("*.dem"), None)

if DEMO_PATH is None:
    raise FileNotFoundError("No .dem file found in the specified directory.")

print("Resolved path:", DEMO_PATH)
print("Exists:", DEMO_PATH.exists())


def load_demo(path: Path):
    path = path.resolve()

    if not path.exists():
        raise FileNotFoundError(f"Demo not found: {path}")

    print(f"[Demo] Loading: {path}")

    demo = Demo(path=str(path))
    demo.parse()

    print("[Demo] Parsing complete.")
    return demo

def inspect_demo_attributes(demo):
    print("\n=== AVAILABLE DEMO ATTRIBUTES ===")

    attrs = dir(demo)

    # filter useful ones
    public_attrs = [a for a in attrs if not a.startswith("_")]

    for attr in sorted(public_attrs):
        try:
            value = getattr(demo, attr)

            if isinstance(value, list):
                print(f"{attr:20} -> list ({len(value)} items)")
            elif isinstance(value, dict):
                print(f"{attr:20} -> dict ({len(value)} keys)")
            else:
                print(f"{attr:20} -> {type(value).__name__}")

        except Exception as e:
            print(f"{attr:20} -> ERROR: {e}")

demo = load_demo(DEMO_PATH)

inspect_demo_attributes(demo)

Resolved path: D:\Git_repo\Internomat\test_demos\2026-03-20_21-57-48_3_de_mills_team_Sindilia_vs_team_Dottersack.dem
Exists: True
[Demo] Loading: D:\Git_repo\Internomat\test_demos\2026-03-20_21-57-48_3_de_mills_team_Sindilia_vs_team_Dottersack.dem
[Demo] Parsing complete.

=== AVAILABLE DEMO ATTRIBUTES ===
bomb                 -> DataFrame
compress             -> method
damages              -> DataFrame
default_events       -> list (20 items)
detected_events      -> list (42 items)
events               -> dict (21 keys)
footsteps            -> DataFrame
grenades             -> DataFrame
header               -> dict (12 keys)
in_play_ticks        -> Series
inferno_duration     -> float
infernos             -> DataFrame
kills                -> DataFrame
parse                -> method
parse_events         -> method
parse_grenades       -> method
parse_header         -> method
parse_ticks          -> method
parser               -> DemoParser
path                 -> WindowsPath
player_round

In [11]:
import pandas as pd

# PARSER
def parse_demo_full(demo):
    print("\n[Demo] Building structured tables (explicit)...")

    data = {}

    # --- HEADER ---
    data["header"] = demo.header

    # --- KNOWN TABLES ---
    table_map = {
        "rounds": "rounds",
        "kills": "kills",
        "damages": "damages",
        "grenades": "grenades",
        "shots": "shots",
        "footsteps": "footsteps",
        "smokes": "smokes",
        "infernos": "infernos",
        "bomb": "bomb",
        "ticks": "ticks",
        "player_round_totals": "player_round_totals",
        "server_cvars": "server_cvars",
    }

    for key, attr in table_map.items():
        try:
            value = getattr(demo, attr, None)

            if value is None:
                print(f"[NULL] {key}")
                data[key] = None
                continue

            # force into DataFrame if possible
            if isinstance(value, pd.DataFrame):
                df = value
            elif isinstance(value, list):
                df = pd.DataFrame(value)
            else:
                df = value  

            data[key] = df

            if isinstance(df, pd.DataFrame):
                print(f"[OK]   {key:22} -> {len(df)} rows")
            else:
                print(f"[OK]   {key:22} -> {type(df).__name__}")

        except Exception as e:
            print(f"[FAIL] {key}: {e}")
            data[key] = None

    print("\n[Demo] Total collected:", len(data))

    return data


data = parse_demo_full(demo)
# print frame 
print("\n=== DEMO DATA STRUCTURE ===\n")
print (data)


[Demo] Building structured tables (explicit)...
[OK]   rounds                 -> DataFrame
[OK]   kills                  -> DataFrame
[OK]   damages                -> DataFrame
[OK]   grenades               -> DataFrame
[OK]   shots                  -> DataFrame
[OK]   footsteps              -> DataFrame
[OK]   smokes                 -> DataFrame
[OK]   infernos               -> DataFrame
[OK]   bomb                   -> DataFrame
[OK]   ticks                  -> DataFrame
[OK]   player_round_totals    -> DataFrame
[OK]   server_cvars           -> DataFrame

[Demo] Total collected: 13

=== DEMO DATA STRUCTURE ===

{'header': {'demo_version_name': 'valve_demo_2', 'server_name': 'MatchZy | team_Sindilia vs team_Dottersack', 'map_name': 'de_mills', 'fullpackets_version': '2', 'game_directory': '/home/server18681/cs2/game/csgo', 'demo_version_guid': '8e9d71ab-04a1-4c01-bb61-acfede27c046', 'demo_file_stamp': 'PBDEMS2\x00', 'client_name': 'SourceTV Demo', 'allow_clientside_entities': True, 

In [5]:
def build_player_index(data):
    df = data["player_round_totals"]

    players = (
        df
        .filter(df["side"] == "all")
        .unique(subset=["steamid"])
    )

    print("[Players]", players.height)
    display(players)

    return players


players = build_player_index(data)

[Players] 10


name,steamid,side,n_rounds
str,u64,str,u32
"""blu""",76561198004039473,"""all""",24
"""Couchyyy1337""",76561198034161077,"""all""",24
"""Soulcrusher""",76561198354578034,"""all""",24
"""Dottersack""",76561198240903150,"""all""",24
"""Tundra ツ""",76561197987271352,"""all""",24
"""Captain Blake""",76561198051680113,"""all""",24
"""-Zuria-""",76561198010796900,"""all""",24
"""KeTaNoS""",76561198045998047,"""all""",24
"""Dude""",76561197970397706,"""all""",24


In [18]:
import polars as pl

def build_player_stats(data):
    print("[Stats] Building...")

    # --- SOURCE TABLES ---
    kills = data["kills"]
    damages = data["damages"]
    shots = data["shots"]
    rounds = data["rounds"]
    players_df = data["player_round_totals"]

    # --- PLAYER INDEX (clean + unique) ---
    players = (
        players_df
        .filter(pl.col("side") == "all")
        .select(["steamid", "name"])
        .unique(subset=["steamid"])
    )

    # --- NORMALIZE EVENTS ---
    kills_norm = kills.select([
        pl.col("attacker_steamid").alias("steamid"),
        pl.lit(1).alias("kills"),
        pl.lit(0).alias("deaths"),
        pl.lit(0).alias("damage"),
        pl.lit(0).alias("shots"),
    ])

    deaths_norm = kills.select([
        pl.col("victim_steamid").alias("steamid"),
        pl.lit(0).alias("kills"),
        pl.lit(1).alias("deaths"),
        pl.lit(0).alias("damage"),
        pl.lit(0).alias("shots"),
    ])

    damage_norm = damages.select([
        pl.col("attacker_steamid").alias("steamid"),
        pl.lit(0).alias("kills"),
        pl.lit(0).alias("deaths"),
        pl.col("dmg_health_real").alias("damage"),
        pl.lit(0).alias("shots"),
    ])

    shots_norm = shots.select([
        pl.col("player_steamid").alias("steamid"),
        pl.lit(0).alias("kills"),
        pl.lit(0).alias("deaths"),
        pl.lit(0).alias("damage"),
        pl.lit(1).alias("shots"),
    ])

    # --- CONCAT EVENTS ---
    all_events = pl.concat([
        kills_norm,
        deaths_norm,
        damage_norm,
        shots_norm
    ])

    # --- AGGREGATE ---
    stats = (
        all_events
        .group_by("steamid")
        .agg([
            pl.sum("kills"),
            pl.sum("deaths"),
            pl.sum("damage"),
            pl.sum("shots"),
        ])
    )

    # --- DERIVED METRICS ---
    n_rounds = rounds.height

    stats = stats.with_columns([
        (pl.col("kills") / pl.when(pl.col("deaths") == 0).then(1).otherwise(pl.col("deaths"))).alias("kd"),
        (pl.col("damage") / n_rounds).alias("adr"),
        (pl.col("kills") / pl.when(pl.col("shots") == 0).then(1).otherwise(pl.col("shots"))).alias("accuracy")
    ])

    # --- JOIN PLAYER NAMES ---
    stats = stats.join(players, on="steamid", how="left")

    # --- COLUMN ORDER ---
    stats = stats.select([
        "steamid",
        "name",
        "kills",
        "deaths",
        "damage",
        "shots",
        "kd",
        "adr",
        "accuracy",
    ])

    # --- SORT ---
    stats = stats.sort("kills", descending=True)

    print(f"[Players] {players.height}")
    print("[Stats] Done")

    display(stats)

    return stats


player_stats = build_player_stats(data)

[Stats] Building...
[Players] 10
[Stats] Done


steamid,name,kills,deaths,damage,shots,kd,adr,accuracy
u64,str,i32,i32,i32,i32,f64,f64,f64
76561198004039473,"""blu""",31,15,2996,466,2.066667,124.833333,0.066524
76561198045998047,"""KeTaNoS""",29,16,2666,382,1.8125,111.083333,0.075916
76561197987271352,"""Tundra ツ""",21,15,1877,427,1.4,78.208333,0.04918
76561198095760902,"""Sindilia""",20,19,2613,309,1.052632,108.875,0.064725
76561198051680113,"""Captain Blake""",16,20,1556,264,0.8,64.833333,0.060606
…,…,…,…,…,…,…,…,…
76561198354578034,"""Soulcrusher""",14,18,1355,375,0.777778,56.458333,0.037333
76561198034161077,"""Couchyyy1337""",14,20,1957,419,0.7,81.541667,0.033413
76561198010796900,"""-Zuria-""",10,21,1384,408,0.47619,57.666667,0.02451
